# Baseline 3 — Widar_LeNet / CNN-5 (SenseFi)

**Source:** Yang et al., *SenseFi: A Library and Benchmark on Deep-Learning-Empowered
WiFi Human Sensing*, Patterns (Cell Press), 2023.  
GitHub: https://github.com/xyanchen/WiFi-CSI-Sensing-Benchmark — `widar_model.py`

**Architecture (exact from SenseFi, one adaptation):**
```
Input  (B, T, 20, 20)  — T time frames treated as 2D channels
  Conv2d(T→32,  k=6, stride=2) → ReLU   → (B, 32,  8,  8)
  Conv2d(32→64, k=3, stride=1) → ReLU   → (B, 64,  6,  6)
  Conv2d(64→96, k=3, stride=1) → ReLU   → (B, 96,  4,  4)
  Flatten                                → (B, 1536)
  Linear(1536→128) → ReLU → Linear(128→6)
```

**Adaptation:** SenseFi uses T=22 frames. Our preprocessing uses T=20.
We set `in_channels=20` instead of 22. All other layers are identical.
This must be stated clearly in the paper.

**Protocol:** 3-fold Leave-One-Room-Out (LORO), standard-6 gestures.  
**Runtime:** ~10–20 min total on Colab **T4 GPU**.

## 1 — Colab Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_DIR = '/content/pi-ssl'
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Fatima112299/pi-ssl.git {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2 — Configuration

In [ ]:
NPZ_PATH      = '/content/drive/MyDrive/pi-ssl/preprocessed.npz'

EPOCHS        = 100
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2
RANDOM_SEED   = 42
LABELED_RATIO = 0.25
NUM_CLASSES   = 6
T_FRAMES      = 20   # our T; SenseFi original uses 22

print(f'NPZ_PATH : {NPZ_PATH}')
print(f'T_FRAMES : {T_FRAMES}  (SenseFi original: 22 — adapted to our preprocessing)')

## 3 — Imports

In [ ]:
import sys, os
REPO_DIR = '/content/pi-ssl'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if not os.path.exists(REPO_DIR):
    raise RuntimeError('Repo not found — run the clone cell first.')

import time, json, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report

from src.data.bvp_dataset import load_npz, BVPDataset
from src.data.splits import make_loeo_splits
from src.data.widar3_dataset import ID_TO_GESTURE

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU. Enable T4 GPU in Runtime → Change runtime type.')
print('All imports OK.')

## 4 — Load Data

In [ ]:
print('Loading preprocessed.npz ...')
t0 = time.time()
npz_data, file_list = load_npz(NPZ_PATH)
print(f'  Samples   : {len(file_list)}')
print(f'  BVP shape : {npz_data["bvp"].shape}  {npz_data["bvp"].dtype}')
print(f'  Load time : {time.time()-t0:.1f}s')

## 5 — Widar_LeNet Model

Exact architecture from SenseFi `widar_model.py`, adapted for T=20.

```python
# Original SenseFi code (T=22):
# nn.Conv2d(22, 32, 6, stride=2)

# Our adaptation (T=20):
# nn.Conv2d(20, 32, 6, stride=2)
# All other layers identical.
```

In [ ]:
class Widar_LeNet(nn.Module):
    """
    LeNet-style CNN from SenseFi (Yang et al., Patterns 2023).
    Also referred to as CNN-5 (3 conv + 2 linear = 5 layers).

    Original input: (batch, 22, 20, 20) — T=22 frames as 2D channels.
    Our input     : (batch, 20, 20, 20) — T=20 frames as 2D channels.
    Spatial feature maps: identical to original (floor arithmetic unaffected).

    Source: https://github.com/xyanchen/WiFi-CSI-Sensing-Benchmark/blob/main/widar_model.py
    """
    def __init__(self, num_classes=6, t_frames=20):
        super().__init__()
        self.encoder = nn.Sequential(
            # Input: (batch, t_frames, 20, 20)
            # SenseFi original: Conv2d(22, 32, 6, stride=2)
            nn.Conv2d(t_frames, 32, kernel_size=6, stride=2),  # → (32, 8, 8)
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),         # → (64, 6, 6)
            nn.ReLU(True),
            nn.Conv2d(64, 96, kernel_size=3, stride=1),         # → (96, 4, 4)
            nn.ReLU(True),
        )
        self.fc = nn.Sequential(
            nn.Linear(96 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x from BVPDataset: (B, 1, 20, 20, 20)
        # Squeeze channel dim and treat T as 2D spatial channels
        x = x.squeeze(1)          # (B, 20, 20, 20) = (B, T, H, W)
        x = self.encoder(x)       # (B, 96, 4, 4)
        x = x.view(x.size(0), -1) # (B, 1536)
        return self.fc(x)         # (B, num_classes)


# Shape check
set_seed(RANDOM_SEED)
_m = Widar_LeNet(num_classes=NUM_CLASSES, t_frames=T_FRAMES).to(DEVICE)
_x = torch.zeros(2, 1, 20, 20, 20).to(DEVICE)   # BVPDataset format
_o = _m(_x)
print(f'Widar_LeNet output shape : {_o.shape}  (expect [2, {NUM_CLASSES}])')
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Trainable params         : {n_params:,}')
del _m, _x, _o

## 6 — Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for bvp, labels in loader:
        bvp, labels = bvp.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(bvp), labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * len(labels)
        total_correct += (model(bvp).argmax(1) == labels).sum().item()
        total_n       += len(labels)
    return total_loss / total_n, total_correct / total_n


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for bvp, labels in loader:
        bvp    = bvp.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(bvp)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * len(labels)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_n       += len(labels)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for bvp, labels in loader:
        preds = model(bvp.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


def run_fold_setting(tag, train_files, test_files, npz_data, device,
                     epochs, batch_size, lr, weight_decay, seed, num_classes, t_frames):
    set_seed(seed)
    print(f'\n  [{tag}]  train={len(train_files)}  test={len(test_files)}')

    train_ds = BVPDataset(train_files, npz_data)
    test_ds  = BVPDataset(test_files,  npz_data)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'),
                              drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size*2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))

    model     = Widar_LeNet(num_classes=num_classes, t_frames=t_frames).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    t_start = time.time()
    for epoch in range(1, epochs + 1):
        loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        scheduler.step()
        if epoch % 20 == 0 or epoch == 1:
            print(f'    Epoch {epoch:>3}/{epochs}  loss={loss:.4f}  '
                  f'train_acc={100*train_acc:.1f}%  elapsed={time.time()-t_start:.0f}s')

    train_time = time.time() - t_start
    y_pred, y_test = evaluate(model, test_loader, device)
    acc = accuracy_score(y_test, y_pred) * 100
    print(f'    Train time : {train_time:.1f}s')
    print(f'    Test acc   : {acc:.2f}%')

    labels_present = sorted(set(y_test.tolist()))
    class_names    = [ID_TO_GESTURE[i] for i in labels_present]
    report = classification_report(y_test, y_pred, labels=labels_present,
                                   target_names=class_names, digits=3, zero_division=0)
    print('    Per-class breakdown:')
    for line in report.strip().split('\n'):
        print(f'      {line}')

    return {'accuracy': acc, 'train_time': train_time, 'y_pred': y_pred, 'y_test': y_test}


print('Utilities defined.')

## 7 — Run 3-Fold LORO Evaluation

> **Runtime:** ~3–5 min per model on T4 GPU. 6 models ≈ 20–30 min total.

In [ ]:
FOLD_ROOMS   = {0: 'Office', 1: 'Classroom', 2: 'Hall'}
fold_results = {}
t_total      = time.time()

for fold in range(3):
    print(f'\n{"="*62}')
    print(f'FOLD {fold}  —  test room: {FOLD_ROOMS[fold]}')
    print(f'{"="*62}')

    _, labeled, unlabeled, test = make_loeo_splits(
        bvp_root=None, fold=fold,
        labeled_ratio=LABELED_RATIO, seed=RANDOM_SEED,
        file_list=file_list
    )
    full_train = labeled + unlabeled

    print(f'  Labeled train (25%) : {len(labeled)}')
    print(f'  Full train   (100%) : {len(full_train)}')
    print(f'  Test                : {len(test)}')

    fold_results[fold] = {}

    fold_results[fold]['25%_labeled'] = run_fold_setting(
        '25%_labeled', labeled, test, npz_data, DEVICE,
        EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, RANDOM_SEED, NUM_CLASSES, T_FRAMES
    )
    fold_results[fold]['100%_labeled'] = run_fold_setting(
        '100%_labeled', full_train, test, npz_data, DEVICE,
        EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, RANDOM_SEED, NUM_CLASSES, T_FRAMES
    )

print(f'\nTotal wall time: {(time.time()-t_total)/60:.1f} min')

## 8 — Summary Table

In [ ]:
accs_25  = [fold_results[f]['25%_labeled']['accuracy']  for f in range(3)]
accs_100 = [fold_results[f]['100%_labeled']['accuracy'] for f in range(3)]
mean_25, std_25   = np.mean(accs_25),  np.std(accs_25)
mean_100, std_100 = np.mean(accs_100), np.std(accs_100)

print('\n' + '='*65)
print(f'{"RESULTS — Widar_LeNet / CNN-5 (SenseFi)": ^65}')
print('='*65)
print(f'{"Fold":<6} {"Test Room":<12} {"LeNet 25% labels":>18} {"LeNet 100% labels":>19}')
print('-'*65)
for f in range(3):
    print(f'{f:<6} {FOLD_ROOMS[f]:<12} '
          f'{fold_results[f]["25%_labeled"]["accuracy"]:>17.2f}%  '
          f'{fold_results[f]["100%_labeled"]["accuracy"]:>17.2f}%')
print('-'*65)
print(f'{"Mean":<18} {mean_25:>17.2f}%  {mean_100:>17.2f}%')
print(f'{"Std":<18}  {std_25:>16.2f}%   {std_100:>16.2f}%')
print('='*65)
print()
print('Context:')
print(f'  SVM (ours)  25% labels  : 62.59% ± 8.85%')
print(f'  SVM (ours) 100% labels  : 69.03% ± 7.76%')
print(f'  LeNet/CNN-5 25%  labels : {mean_25:.2f}% ± {std_25:.2f}%')
print(f'  LeNet/CNN-5 100% labels : {mean_100:.2f}% ± {std_100:.2f}%')
print(f'  PI-SSL target (25%)     : >89.50%')

## 9 — Save Results

In [ ]:
RESULTS_PATH = '/content/drive/MyDrive/pi-ssl/results_lenet_baseline.json'

summary = {
    'method'         : 'Widar_LeNet (SenseFi CNN-5)',
    'source_paper'   : 'Yang et al., Patterns 2023',
    'source_code'    : 'https://github.com/xyanchen/WiFi-CSI-Sensing-Benchmark',
    'adaptation'     : 'in_channels changed from 22 to 20 to match our T_TARGET=20 preprocessing',
    'epochs'         : EPOCHS,
    'batch_size'     : BATCH_SIZE,
    'lr'             : LR,
    'weight_decay'   : WEIGHT_DECAY,
    'labeled_ratio'  : LABELED_RATIO,
    'seed'           : RANDOM_SEED,
    'folds': {
        str(f): {
            'test_room'        : FOLD_ROOMS[f],
            'acc_25pct'        : fold_results[f]['25%_labeled']['accuracy'],
            'acc_100pct'       : fold_results[f]['100%_labeled']['accuracy'],
            'train_time_25pct' : fold_results[f]['25%_labeled']['train_time'],
            'train_time_100pct': fold_results[f]['100%_labeled']['train_time'],
        } for f in range(3)
    },
    'mean_acc_25pct'  : float(mean_25),
    'std_acc_25pct'   : float(std_25),
    'mean_acc_100pct' : float(mean_100),
    'std_acc_100pct'  : float(std_100),
}

with open(RESULTS_PATH, 'w') as fp:
    json.dump(summary, fp, indent=2)

print(f'Results saved to {RESULTS_PATH}')
print(json.dumps(summary, indent=2))

---
## Paper Note

In your paper write:

> "We adopt the Widar_LeNet (CNN-5) architecture from SenseFi [Yang et al., 2023],
> adapting the first convolutional layer input channels from 22 to 20 to match
> our temporal standardisation (T=20). All other layers are identical to the
> original implementation. We re-evaluate under our 3-fold LORO protocol."

**Next notebook:** `baseline_cnngru.ipynb` — Widar_CNN_GRU from the same SenseFi repo
(architecture matches Widar3.0 TPAMI paper's recognition model).